In [2]:
# packages for website data scraping
import pandas as pd
import numpy as np
from selenium import webdriver
from lxml import etree
import time
import json
import os
from urllib.request import urlopen, Request
from bs4 import BeautifulSoup
import re
import threading
import concurrent
import random

In [3]:
def get_page_data(url):
    
    options = webdriver.ChromeOptions()
    options.add_argument("--incognito")
    options.add_argument("--disable-blink-features=AutomationControlled")


    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)
        time.sleep(40)
        page_html = driver.page_source
        soup = BeautifulSoup(page_html, 'html.parser')

        # kill script and design eleements
        for script in soup(["script", "style"]):
            script.extract()    # rip it out
        
        # get text
        text = soup.get_text()

        # break into lines and remove leading and trailing space on each
        lines = (line.strip() for line in text.splitlines())
        # break multi-headlines into a line each
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        # drop blank lines
        text = '\n'.join(chunk for chunk in chunks if chunk)

        # also get all tables
        soup = BeautifulSoup(page_html, 'html.parser')
        tables = soup.find_all('table')
        table_dfs = []
        for table in tables:
            table_html = str(table)
            table_soup = BeautifulSoup(table_html, 'html.parser')
            table_df = pd.read_html(str(table_soup))[0]
            table_dfs.append(table_df)
        
        # return the text and tables
        return text, table_dfs
        
    except:
        print("webdriver failed to get url",url)
        return None, None
    
    finally:
        driver.quit()

In [4]:
def get_page_data_dump(url, year, team):

    print("getting data for",team,year)

    options = webdriver.ChromeOptions()
    options.add_argument("--incognito")
    options.add_argument("--disable-blink-features=AutomationControlled")


    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)
        time.sleep(80)
        page_html = driver.page_source
        soup = BeautifulSoup(page_html, 'html.parser')

        # kill script and design eleements
        for script in soup(["script", "style"]):
            script.extract()    # rip it out
        
        # get text
        text = soup.get_text()

        # break into lines and remove leading and trailing space on each
        lines = (line.strip() for line in text.splitlines())
        # break multi-headlines into a line each
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        # drop blank lines
        text = '\n'.join(chunk for chunk in chunks if chunk)

        # safeguard against cloudflare -> if text under 100 lines, then likely cloudflare, return False
        if len(text.splitlines()) < 100:
            return False

        # also get all tables
        soup = BeautifulSoup(page_html, 'html.parser')
        tables = soup.find_all('table')
        table_dfs = []
        for table in tables:
            table_html = str(table)
            table_soup = BeautifulSoup(table_html, 'html.parser')
            table_df = pd.read_html(str(table_soup))[0]
            table_dfs.append(table_df)
        
        # dump into two files, one for text and one for tables
        text_filename = f"data/{team}_{year}_text.txt"
        with open(text_filename, 'w', encoding='utf-8') as f:
            f.write(text)
        
        tables_filename = f"data/{team}_{year}_tables.pkl"
        with open(tables_filename, 'wb') as f:
            pd.to_pickle(table_dfs, f)
        
        return True

    except:
        print("webdriver failed to get url",url)
        return False
    
    finally:
        driver.quit()

In [5]:
def try_fetch(url, year, team):
    # sleep a random amount of time between 0 and 10 seconds to avoid being blocked by cloudflare
    time.sleep(random.uniform(0, 10))
    success = get_page_data_dump(url, year, team)
    while not success:
        time.sleep(10)
        success = get_page_data_dump(url, year, team)

In [6]:
# URLs to fetch

# Base URL
# url = "https://www.pro-football-reference.com/teams/<team>/<year>.htm"

# years to fetch
years = [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

# teams to fetch
teams = ["atl", "buf", "car", "chi", "cin", "cle", "clt", "crd", "dal", "den", "det", "gnb", "htx", "jax", "kan", "mia", "min", "nor", "nwe", "nyg", "nyj", "oti", "phi", "pit", "rai", "ram", "rav", "sdg", "sea", "sfo", "tam", "was"]

# exceptions (Texans only joined the league in 2002)
exceptions = {("htx", 1999), ("htx", 2000), ("htx", 2001)}

In [ ]:
# Fetch data for all combinations of teams and years to be dumped into files, use threading to speed up the process, because there is long sleep time for each request, and we can do multiple requests in parallel. Use multiprocessing to create a pool of workers to fetch data in parallel.

# 8 workers using threading
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    futures = []
    for year in years:
        for team in teams:

            if (team, year) in exceptions:
                continue

            url = f"https://www.pro-football-reference.com/teams/{team}/{year}.htm"
            futures.append(executor.submit(try_fetch, url, year, team))
    
    # wait for all futures to complete
    concurrent.futures.wait(futures)

In [7]:
# Pattern 1: Team Record for the season

# Example:
# Record: 14-3-0, 1st in
# NFC West Division
# (Schedule and Results)
# Coach:
# Mike Macdonald
# Points For: 483 (28.4/g) 3rd of 32
# Points Against: 292 (17.2/g) 1st of 32
# Expected W-L: 13.0-4.0
# SRS: 12.85 (1st of 32),
# SOS: 1.61

# Second Example:
# Record: 10-6-0, 1st in
# NFC East Division
# (Schedule and Results)
# Coach:
# Norv Turner (10-6-0)
# Points For: 443 (27.7/g) 2nd of 31
# Points Against: 377 (23.6/g) 24th of 31
# Expected W-L: 9.5-6.5
# SRS: 2.88 (12th of 31),
# SOS: -1.25

# Third Example:
# Record: 10-4-3, 4th in
# NFC East Division
# (Schedule and Results)
# Coach:
# Bill Belichick (10-4-3) and
# Mike McCarthy (10-2-1)
# Ben McAdoo (0-2-2)
# Points For: 400 (25.0/g) 5th of 32
# Points Against: 350 (21.9/g) 20th of 32
# Expected W-L: 8.5-5.5
# SRS: -.34 (17th of 32),
# SOS: .22

# data in the pattern:
# Division Placement,Points For,Points for Per Game,Points For Ranking,Points Against,Points Against Per Game,Points Against Ranking,Expected Wins,Expected Losses,Team Rating SRS,SRS rank,Strength of Schedule

# Pattern:
# Record: %d-%d-%d, <Division Placement> in
# <Division Name> Division
# (Schedule and Results)
# Coach:
# (Coach Info, might be multiple lines, skip)
# Points For: <Points For> (<Points for Per Game>/g) <Points For Ranking> of <total team number>
# Points Against: <Points Against> (<Points Against Per Game>/g) <Points Against Ranking> of <total team number>
# Expected W-L: <Expected Wins>-<Expected Losses>
# SRS: <Team Rating SRS (can be positive or negative, 0.xx numbers might be written as .xx or -.xx, so just .41 or -.41)> (<SRS rank> of <total team number>),
# SOS: <Strength of Schedule (can be positive or negative, 0.xx numbers might be written as .xx or -.xx, so just .41 or -.41)>

# Regex Pattern:
pattern = r"Record: (\d+)-(\d+)-(\d+), (\d+)(?:st|nd|rd|th) in\n([A-Za-z ]+) Division\n\(Schedule and Results\)\nCoach:\n(?s:.*)\nPoints For: (\d+) \(([\d.]+)/g\) (\d+)(?:st|nd|rd|th) of \d+\nPoints Against: (\d+) \(([\d.]+)/g\) (\d+)(?:st|nd|rd|th) of \d+\nExpected W-L: ([\d.]+)-([\d.]+)\nSRS: ([\d.-]+) \((\d+)(?:st|nd|rd|th) of \d+\),\nSOS: ([\d.-]+)"

# Test this pattern on the sample text to see if it can extract the relevant data. If it works, we can then use it to extract the data from the actual webpage.
example_texts = [
    """
Record: 14-3-0, 1st in
NFC West Division
(Schedule and Results)
Coach:
Mike Macdonald
Points For: 483 (28.4/g) 3rd of 32
Points Against: 292 (17.2/g) 1st of 32
Expected W-L: 13.0-4.0
SRS: 12.85 (1st of 32),
SOS: 1.61""",
    """
Record: 10-6-0, 1st in
NFC East Division
(Schedule and Results)
Coach:
Norv Turner (10-6-0)
Points For: 443 (27.7/g) 2nd of 31
Points Against: 377 (23.6/g) 24th of 31
Expected W-L: 9.5-6.5
SRS: 2.88 (12th of 31),
SOS: -1.25""",
    """
Record: 10-4-3, 4th in
NFC East Division
(Schedule and Results)
Coach:
Bill Belichick (10-4-3) and
Mike McCarthy (10-2-1) and
Ben McAdoo (0-2-2)
Points For: 400 (25.0/g) 5th of 32
Points Against: 350 (21.9/g) 20th of 32
Expected W-L: 8.5-5.5
SRS: -.34 (17th of 32),
SOS: .22"""
]

# Perform regex search to fill variables
for example_text in example_texts:
    match = re.search(pattern, example_text)
    if match:
        wins = int(match.group(1))
        losses = int(match.group(2))
        ties = int(match.group(3))
        division_placement = int(match.group(4))
        division_name = match.group(5)
        points_for = int(match.group(6))
        points_for_per_game = float(match.group(7))
        points_for_ranking = int(match.group(8))
        points_against = int(match.group(9))
        points_against_per_game = float(match.group(10))
        points_against_ranking = int(match.group(11))
        expected_wins = float(match.group(12))
        expected_losses = float(match.group(13))
        srs_rating = float(match.group(14))
        srs_rank = int(match.group(15))
        sos_rating = float(match.group(16))

        print(f"Wins: {wins}, Losses: {losses}, Ties: {ties}, Division Placement: {division_placement}, Division Name: {division_name}, Points For: {points_for}, Points For Per Game: {points_for_per_game}, Points For Ranking: {points_for_ranking}, Points Against: {points_against}, Points Against Per Game: {points_against_per_game}, Points Against Ranking: {points_against_ranking}, Expected Wins: {expected_wins}, Expected Losses: {expected_losses}, SRS Rating: {srs_rating}, SRS Rank: {srs_rank}, SOS Rating: {sos_rating}")
    else:
        print("No match found for the example text.")


Wins: 14, Losses: 3, Ties: 0, Division Placement: 1, Division Name: NFC West, Points For: 483, Points For Per Game: 28.4, Points For Ranking: 3, Points Against: 292, Points Against Per Game: 17.2, Points Against Ranking: 1, Expected Wins: 13.0, Expected Losses: 4.0, SRS Rating: 12.85, SRS Rank: 1, SOS Rating: 1.61
Wins: 10, Losses: 6, Ties: 0, Division Placement: 1, Division Name: NFC East, Points For: 443, Points For Per Game: 27.7, Points For Ranking: 2, Points Against: 377, Points Against Per Game: 23.6, Points Against Ranking: 24, Expected Wins: 9.5, Expected Losses: 6.5, SRS Rating: 2.88, SRS Rank: 12, SOS Rating: -1.25
Wins: 10, Losses: 4, Ties: 3, Division Placement: 4, Division Name: NFC East, Points For: 400, Points For Per Game: 25.0, Points For Ranking: 5, Points Against: 350, Points Against Per Game: 21.9, Points Against Ranking: 20, Expected Wins: 8.5, Expected Losses: 5.5, SRS Rating: -0.34, SRS Rank: 17, SOS Rating: 0.22


In [8]:
# Function to extract this pattern from the text of the webpage and return a dataframe with the relevant data. If the pattern is not found, return None.

def find_team_record(text):
    
    pattern = r"Record: (\d+)-(\d+)-(\d+), (\d+)(?:st|nd|rd|th) in\n([A-Za-z ]+) Division\n\(Schedule and Results\)\nCoach:\n(?s:.*)\nPoints For: (\d+) \(([\d.]+)/g\) (\d+)(?:st|nd|rd|th) of \d+\nPoints Against: (\d+) \(([\d.]+)/g\) (\d+)(?:st|nd|rd|th) of \d+\nExpected W-L: ([\d.]+)-([\d.]+)\nSRS: ([\d.-]+) \((\d+)(?:st|nd|rd|th) of \d+\),\nSOS: ([\d.-]+)"

    match = re.search(pattern, text)

    if match:
        df = pd.DataFrame({
            'Wins': [int(match.group(1))],
            'Losses': [int(match.group(2))],
            'Ties': [int(match.group(3))],
            'Division Placement': [int(match.group(4))],
            'Division Name': [match.group(5)],
            'Points For': [int(match.group(6))],
            'Points For Per Game': [float(match.group(7))],
            'Points For Ranking': [int(match.group(8))],
            'Points Against': [int(match.group(9))],
            'Points Against Per Game': [float(match.group(10))],
            'Points Against Ranking': [int(match.group(11))],
            'Expected Wins': [float(match.group(12))],
            'Expected Losses': [float(match.group(13))],
            'SRS Rating': [float(match.group(14))],
            'SRS Rank': [int(match.group(15))],
            'SOS Rating': [float(match.group(16))]
        })

        # Add empty columns at start for year and team
        df.insert(0, 'Year', np.nan)
        df.insert(1, 'Team', np.nan)
        return df


In [9]:
# Pattern 2: Team Record for the season

# Since this is tabular data, we should be able to find it in the tables we extracted from the webpage.

# Print tables from one example to find the right one
table_dfs = pd.read_pickle("data/atl_2023_tables.pkl")

# for i, table_df in enumerate(table_dfs):
#     print(f"Table {i}:")
#     print(table_df)
#     print("\n")

# Okay, it'S table 11 in this example, open this one separately and see what it looks like. We can then use this to extract the data we want.
team_record_df = table_dfs[11]
team_record_df

# Get the column names, so we can identify the table we want to extract. We can then use this to extract the data we want.
team_record_df.columns

# After extraction, we want to clean up the column names so they are more readable
col_names = team_record_df.columns

# Get rid of the multi-index and combine the column names into a single level and turn them into strings
col_names = ['_'.join(col).strip() for col in col_names]

# Get rid of whitespace, (, ), ', :, the word Unnamed, and any version of <number>_level_<number>, replace leading and trailing whitespace with nothing, and replace any remaining whitespace with underscores
col_names = pd.Series(col_names)
col_names = col_names.str.strip()
col_names = col_names.str.replace(r"[\(\)\'\:]", "", regex=True)
col_names = col_names.str.replace(r"Unnamed", "", regex=True)
col_names = col_names.str.replace(r"\d+_level_\d+", "", regex=True)
col_names = col_names.str.strip()
col_names = col_names.str.replace(r"\s+", "_", regex=True)

team_record_df.columns = col_names

team_record_df

# Hardcode list of column names
column_names = ['Week', 'Day', 'Date', 'Time', 'Boxscore', 'Result', 'OT', 'Record', 'Home_Away', 'Opponent', 'Score_Tm', 'Score_Opp', 'Offense_1stD', 'Offense_TotYd', 'Offense_PassY', 'Offense_RushY', 'Offense_TO', 'Defense_1stD', 'Defense_TotYd', 'Defense_PassY', 'Defense_RushY', 'Defense_TO', 'Expected_Points_Offense', 'Expected_Points_Defense', 'Expected_Points_Sp_Tms']

team_record_df.columns = column_names

team_record_df

,Week,Day,Date,Time,Boxscore,Result,OT,Record,Home_Away,Opponent,...,Offense_RushY,Offense_TO,Defense_1stD,Defense_TotYd,Defense_PassY,Defense_RushY,Defense_TO,Expected_Points_Offense,Expected_Points_Defense,Expected_Points_Sp_Tms
0,1,Sun,September 10,1:00PM ET,boxscore,W,NaN,1-0,NaN,Carolina Panthers,...,130.0,NaN,20.0,281.0,127.0,154.0,3.0,-2.92,16.61,-0.20
1,2,Sun,September 17,1:00PM ET,boxscore,W,NaN,2-0,NaN,Green Bay Packers,...,211.0,1.0,17.0,224.0,140.0,84.0,NaN,13.12,-7.85,-0.97
2,3,Sun,September 24,1:00PM ET,boxscore,L,NaN,2-1,@,Detroit Lions,...,44.0,1.0,18.0,358.0,243.0,115.0,1.0,-15.09,-1.31,0.59
3,4,Sun,October 1,9:30AM ET,boxscore,L,NaN,2-2,@,Jacksonville Jaguars,...,127.0,3.0,16.0,300.0,195.0,105.0,NaN,-15.78,-0.72,0.86
4,5,Sun,October 8,1:00PM ET,boxscore,W,NaN,3-2,NaN,Houston Texans,...,96.0,2.0,17.0,313.0,249.0,64.0,NaN,6.01,-2.73,-1.01
5,6,Sun,October 15,1:00PM ET,boxscore,L,NaN,3-3,NaN,Washington Commanders,...,106.0,3.0,13.0,193.0,121.0,72.0,NaN,-5.68,0.45,-3.28
6,7,Sun,October 22,1:00PM ET,boxscore,W,NaN,4-3,@,Tampa Bay Buccaneers,...,156.0,3.0,21.0,329.0,256.0,73.0,2.0,-0.80,2.11,1.89
7,8,Sun,October 29,1:00PM ET,boxscore,L,NaN,4-4,@,Tennessee Titans,...,140.0,1.0,17.0,375.0,226.0,149.0,1.0,-3.34,-4.14,-0.17
8,9,Sun,November 5,1:00PM ET,boxscore,L,NaN,4-5,NaN,Minnesota Vikings,...,110.0,2.0,23.0,363.0,217.0,146.0,2.0,-3.40,-1.90,3.43
9,10,Sun,November 12,4:05PM ET,boxscore,L,NaN,4-6,@,Arizona Cardinals,...,184.0,NaN,20.0,352.0,230.0,122.0,1.0,4.70,-0.59,-6.18


In [10]:
# Function to extract this table from the list of tables we extracted from the webpage and return a dataframe of the data. If the table is not found, raise an exception.

def extract_weekly_scores(table_dfs):

    table_df = table_dfs[11]

    # Get rid of the multi-index and combine the column names into a single level and turn them into strings
    table_df.columns = ['_'.join(col).strip() for col in table_df.columns]

    # Check if the table has the right dimensions, if not (table_df.shape[1] == 26 and table_df.shape[0] > 10 and table_df.shape[0] < 23)
    if table_df.shape[1] != 25 or table_df.shape[0] <= 10 or table_df.shape[0] >= 26:
        raise ValueError("Table not found in tables")
    
    # Replace the column names with the hardcoded list of column names
    column_names = ['Week', 'Day', 'Date', 'Time', 'Boxscore', 'Result', 'OT', 'Record', 'Home_Away', 'Opponent', 'Score_Tm', 'Score_Opp', 'Offense_1stD', 'Offense_TotYd', 'Offense_PassY', 'Offense_RushY', 'Offense_TO', 'Defense_1stD', 'Defense_TotYd', 'Defense_PassY', 'Defense_RushY', 'Defense_TO', 'Expected_Points_Offense', 'Expected_Points_Defense', 'Expected_Points_Sp_Tms']

    table_df.columns = column_names

    # Add empty columns at start for year and team
    table_df.insert(0, 'Year', np.nan)
    table_df.insert(1, 'Team', np.nan)

    return table_df


In [10]:
# For each team/year combination try to extract the team record and weekly score. Compile a list of combinations who fail one or both

refetch_list = []

for year in years:
    for team in teams:

        if (team, year) in exceptions:
            continue

        text_filename = f"data/{team}_{year}_text.txt"
        tables_filename = f"data/{team}_{year}_tables.pkl"

        # Check if the files exist
        if not os.path.exists(text_filename) or not os.path.exists(tables_filename):
            print(f"Files for {team} {year} do not exist, skipping")
            refetch_list.append((team, year))
            continue

        # Load the text and tables
        with open(text_filename, 'r', encoding='utf-8') as f:
            text = f.read()
        
        table_dfs = pd.read_pickle(tables_filename)

        # Try to extract the team record
        try:
            team_record_df = find_team_record(text)
            if team_record_df is None:
                print(f"Team record not found for {team} {year}")
                refetch_list.append((team, year))
            else:
                team_record_df['Year'] = year
                team_record_df['Team'] = team
        except Exception as e:
            refetch_list.append((team, year))
            print(f"Error extracting team record for {team} {year}: {e}")

        # Try to extract the weekly scores
        try:
            weekly_scores_df = extract_weekly_scores(table_dfs)
            weekly_scores_df['Year'] = year
            weekly_scores_df['Team'] = team
        except Exception as e:
            refetch_list.append((team, year))
            print(f"Error extracting weekly scores for {team} {year}: {e}")

KeyboardInterrupt: 

In [ ]:
# For combinations that we could not extract from, refetch those from the website

# Get rid of duplicates in refetch list, to make sure no combinations are in there twice
refetch_list = list(set(refetch_list))

# Threading to speed up refetching
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    futures = []
    for team, year in refetch_list:
        url = f"https://www.pro-football-reference.com/teams/{team}/{year}.htm"
        futures.append(executor.submit(try_fetch, url, year, team))
    
    # wait for all futures to complete
    concurrent.futures.wait(futures)

In [11]:
# For each team, combine team record and weekly scores into a single dataframe, and then save it to a csv file. For both, also save one large file with all teams and years combined. Sort these first by year, then by team

# Make dataframes to hold all data
yearly_team_record_all = pd.DataFrame(columns=['Year', 'Team', 'Wins', 'Losses', 'Ties', 'Division Placement', 'Division Name', 'Points For', 'Points For Per Game', 'Points For Ranking', 'Points Against', 'Points Against Per Game', 'Points Against Ranking', 'Expected Wins', 'Expected Losses', 'SRS', 'SRS Rank', 'SOS'])
weekly_scores_all = pd.DataFrame(columns=['Year', 'Team', 'Week', 'Day', 'Date', 'Time', 'Boxscore', 'Result', 'OT', 'Record', 'Home_Away', 'Opponent', 'Score_Tm', 'Score_Opp', 'Offense_1stD', 'Offense_TotYd', 'Offense_PassY', 'Offense_RushY', 'Offense_TO', 'Defense_1stD', 'Defense_TotYd', 'Defense_PassY', 'Defense_RushY', 'Defense_TO', 'Expected_Points_Offense', 'Expected_Points_Defense', 'Expected_Points_Sp_Tms'])

yearly_team_record_by_team = {}
weekly_scores_by_team = {}
for team in teams:
    yearly_team_record_by_team[team] = pd.DataFrame(columns=['Year', 'Team', 'Wins', 'Losses', 'Ties', 'Division Placement', 'Division Name', 'Points For', 'Points For Per Game', 'Points For Ranking', 'Points Against', 'Points Against Per Game', 'Points Against Ranking', 'Expected Wins', 'Expected Losses', 'SRS', 'SRS Rank', 'SOS'])
    weekly_scores_by_team[team] = pd.DataFrame(columns=['Year', 'Team', 'Week', 'Day', 'Date', 'Time', 'Boxscore', 'Result', 'OT', 'Record', 'Home_Away', 'Opponent', 'Score_Tm', 'Score_Opp', 'Offense_1stD', 'Offense_TotYd', 'Offense_PassY', 'Offense_RushY', 'Offense_TO', 'Defense_1stD', 'Defense_TotYd', 'Defense_PassY', 'Defense_RushY', 'Defense_TO', 'Expected_Points_Offense', 'Expected_Points_Defense', 'Expected_Points_Sp_Tms'])

# Extract data from the dumped files and combine them into the dataframes
for year in years:
    for team in teams:

        if (team, year) in exceptions:
            continue

        print(f"Processing {team} ({year})")
        text_filename = f"data/{team}_{year}_text.txt"
        tables_filename = f"data/{team}_{year}_tables.pkl"
        if os.path.exists(text_filename) and os.path.exists(tables_filename):
            with open(text_filename, 'r', encoding='utf-8') as f:
                text = f.read()
            table_dfs = pd.read_pickle(tables_filename)
            team_record_df = find_team_record(text)
            weekly_scores_df = extract_weekly_scores(table_dfs)
            team_record_df['Year'] = year
            team_record_df['Team'] = team
            weekly_scores_df['Year'] = year
            weekly_scores_df['Team'] = team
            yearly_team_record_all = pd.concat([yearly_team_record_all, team_record_df], ignore_index=True)
            weekly_scores_all = pd.concat([weekly_scores_all, weekly_scores_df], ignore_index=True)
            yearly_team_record_by_team[team] = pd.concat([yearly_team_record_by_team[team], team_record_df], ignore_index=True)
            weekly_scores_by_team[team] = pd.concat([weekly_scores_by_team[team], weekly_scores_df], ignore_index=True)
    print(f"Finished processing year {year}")


# Save the dataframes to csv files
yearly_team_record_all.to_csv("data/yearly_team_record_all.csv", index=False)
weekly_scores_all.to_csv("data/weekly_scores_all.csv", index=False)

for team in teams:
    yearly_team_record_by_team[team].to_csv(f"data/yearly_team_record_{team}.csv", index=False)
    weekly_scores_by_team[team].to_csv(f"data/weekly_scores_{team}.csv", index=False)


Processing atl (1999)
Processing buf (1999)
Processing car (1999)
Processing chi (1999)
Processing cin (1999)
Processing cle (1999)
Processing clt (1999)
Processing crd (1999)
Processing dal (1999)
Processing den (1999)
Processing det (1999)


C:\Users\maike\AppData\Local\Temp\ipykernel_17944\3207047005.py:33: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  yearly_team_record_all = pd.concat([yearly_team_record_all, team_record_df], ignore_index=True)
C:\Users\maike\AppData\Local\Temp\ipykernel_17944\3207047005.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  weekly_scores_all = pd.concat([weekly_scores_all, weekly_scores_df], ignore_index=True)
C:\Users\maike\AppData\Local\Temp\ipykernel_17944\3207047005.py:35: FutureWarning: The b

Processing gnb (1999)
Processing jax (1999)
Processing kan (1999)
Processing mia (1999)
Processing min (1999)
Processing nor (1999)
Processing nwe (1999)
Processing nyg (1999)
Processing nyj (1999)


C:\Users\maike\AppData\Local\Temp\ipykernel_17944\3207047005.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  yearly_team_record_by_team[team] = pd.concat([yearly_team_record_by_team[team], team_record_df], ignore_index=True)
C:\Users\maike\AppData\Local\Temp\ipykernel_17944\3207047005.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  weekly_scores_by_team[team] = pd.concat([weekly_scores_by_team[team], weekly_scores_df], ignore_index=True)
C:\Users\maike\AppData\Local\Temp\ipykernel_1794

Processing oti (1999)
Processing phi (1999)
Processing pit (1999)
Processing rai (1999)
Processing ram (1999)
Processing rav (1999)
Processing sdg (1999)
Processing sea (1999)
Processing sfo (1999)
Processing tam (1999)
Processing was (1999)
Finished processing year 1999
Processing atl (2000)
Processing buf (2000)
Processing car (2000)
Processing chi (2000)
Processing cin (2000)
Processing cle (2000)
Processing clt (2000)
Processing crd (2000)
Processing dal (2000)
Processing den (2000)
Processing det (2000)
Processing gnb (2000)
Processing jax (2000)
Processing kan (2000)
Processing mia (2000)
Processing min (2000)
Processing nor (2000)
Processing nwe (2000)
Processing nyg (2000)
Processing nyj (2000)
Processing oti (2000)
Processing phi (2000)
Processing pit (2000)
Processing rai (2000)
Processing ram (2000)
Processing rav (2000)
Processing sdg (2000)
Processing sea (2000)
Processing sfo (2000)
Processing tam (2000)
Processing was (2000)
Finished processing year 2000
Processing atl (

C:\Users\maike\AppData\Local\Temp\ipykernel_17944\3207047005.py:35: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  yearly_team_record_by_team[team] = pd.concat([yearly_team_record_by_team[team], team_record_df], ignore_index=True)
C:\Users\maike\AppData\Local\Temp\ipykernel_17944\3207047005.py:36: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  weekly_scores_by_team[team] = pd.concat([weekly_scores_by_team[team], weekly_scores_df], ignore_index=True)


Processing nor (2002)
Processing nwe (2002)
Processing nyg (2002)
Processing nyj (2002)
Processing oti (2002)
Processing phi (2002)
Processing pit (2002)
Processing rai (2002)
Processing ram (2002)
Processing rav (2002)
Processing sdg (2002)
Processing sea (2002)
Processing sfo (2002)
Processing tam (2002)
Processing was (2002)
Finished processing year 2002
Processing atl (2003)
Processing buf (2003)
Processing car (2003)
Processing chi (2003)
Processing cin (2003)
Processing cle (2003)
Processing clt (2003)
Processing crd (2003)
Processing dal (2003)
Processing den (2003)
Processing det (2003)
Processing gnb (2003)
Processing htx (2003)
Processing jax (2003)
Processing kan (2003)
Processing mia (2003)
Processing min (2003)
Processing nor (2003)
Processing nwe (2003)
Processing nyg (2003)
Processing nyj (2003)
Processing oti (2003)
Processing phi (2003)
Processing pit (2003)
Processing rai (2003)
Processing ram (2003)
Processing rav (2003)
Processing sdg (2003)
Processing sea (2003)
Pr

In [12]:
# We want to know how badly each team needs to win in the next game
# For every week in the regular season, for all seasons, therefore use wins, losses, and ties up to that point in the season to calculate league standings and division standings for each team

# For all combinations of year, team, and week, tally up the wins, losses, and ties for each team up to that point in the season
# For every week, results are all previous weeks results added up, so {0, 0, 0} for week 1, and then for week 2, add the result of week 1, and so on. Therefore, we can just iterate through the weeks in order and keep a running tally of wins, losses, and ties for each team.
weekly_scores_all = pd.read_csv("data/weekly_scores_all.csv")
yearly_team_record_all = pd.read_csv("data/yearly_team_record_all.csv")
score_tallys = pd.DataFrame(columns=['Year', 'Week', 'Team', 'Division', 'Wins', 'Losses', 'Ties'])

for year in years:
    for week in range(1, 19):

        # Check if that week exists in the data, if not, skip it (league expansions, etc.)
        if not ((weekly_scores_all['Year'] == year) & (weekly_scores_all['Week'] == str(week))).any():
            continue

        for team in teams:

            if (team, year) in exceptions:
                continue

            # Get the results for this team up to this week
            valid_week_strings = [str(w) for w in range(1, week)]
            team_results = weekly_scores_all[(weekly_scores_all['Year'] == year) & (weekly_scores_all['Team'] == str(team)) & (weekly_scores_all['Week'].isin(valid_week_strings))]
            wins = team_results['Result'].str.count('W').sum()
            losses = team_results['Result'].str.count('L').sum()
            ties = team_results['Result'].str.count('T').sum()

            # Get the division for this team for this year
            division = yearly_team_record_all[(yearly_team_record_all['Year'] == year) & (yearly_team_record_all['Team'] == team)]['Division Name'].values[0]

            score_tallys = pd.concat([score_tallys, pd.DataFrame({'Year': [year], 'Week': [week], 'Team': [team], 'Division': [division], 'Wins': [int(wins)], 'Losses': [int(losses)], 'Ties': [int(ties)]})], ignore_index=True)

# For each week, calculate the league standings and division standings for each team based on the wins, losses, and ties up to that point in the season. For league standings, sort by wins, then losses, then ties. For division standings, sort by wins, then losses, then ties within each division. Add columns for league rank and division rank to the score_tallys dataframe.
# Get the division for each team from the yearly_team_record_all dataframe for that year
weekly_standings = pd.DataFrame(columns=['Year', 'Week', 'Team', 'Division', 'Wins', 'Losses', 'Ties', 'League Rank', 'Division Rank'])

for year in years:
    for week in range(1, 19):

        # Check if that week exists in the data, if not, skip it (league expansions, etc.)
        if not ((weekly_scores_all['Year'] == year) & (weekly_scores_all['Week'] == str(week))).any():
            continue

        for team in teams:

            if (team, year) in exceptions:
                continue

            # Get the wins, losses, and ties for this team up to this week, and the division for this team for this year
            team_results = score_tallys[(score_tallys['Year'] == year) & (score_tallys['Team'] == team) & (score_tallys['Week'] == week)]
            wins = team_results['Wins'].values[0]
            losses = team_results['Losses'].values[0]
            ties = team_results['Ties'].values[0]
            division = team_results['Division'].values[0]

            # Get the league standings for this week
            league_standings = score_tallys[(score_tallys['Year'] == year) & (score_tallys['Week'] == week)].sort_values(by=['Wins', 'Losses', 'Ties'], ascending=[False, True, True]).reset_index(drop=True)
            league_rank = league_standings[league_standings['Team'] == team].index[0] + 1

            # Get the division standings for this week
            division_standings = score_tallys[(score_tallys['Year'] == year) & (score_tallys['Week'] == week) & (score_tallys['Division'] == division)].sort_values(by=['Wins', 'Losses', 'Ties'], ascending=[False, True, True]).reset_index(drop=True)
            division_rank = division_standings[division_standings['Team'] == team].index[0] + 1

            weekly_standings = pd.concat([weekly_standings, pd.DataFrame({'Year': [year], 'Week': [week], 'Team': [team], 'Division': [division], 'Wins': [wins], 'Losses': [losses], 'Ties': [ties], 'League Rank': [league_rank], 'Division Rank': [division_rank]})], ignore_index=True)
            

# Save the weekly_standings dataframe to a csv file
weekly_standings.to_csv("data/weekly_standings.csv", index=False)
